In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from functools import reduce

In [7]:
spark = SparkSession.builder \
    .appName("IndianFood") \
    .getOrCreate()

In [8]:
df = spark.read.csv("D:\BDA 17\ABD\indian food\indian_food.csv", header=True,inferSchema=True)

In [9]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- diet: string (nullable = true)
 |-- prep_time: integer (nullable = true)
 |-- cook_time: integer (nullable = true)
 |-- flavor_profile: string (nullable = true)
 |-- course: string (nullable = true)
 |-- state: string (nullable = true)



In [12]:
#1. Find out how many unique dishes are present.
df.select(countDistinct("name").alias("unique_dishes")).show()

+-------------+
|unique_dishes|
+-------------+
|          255|
+-------------+



In [18]:
#2. Which state has more dishes? 

df.groupBy("state")\
  .count()\
  .orderBy(desc("count"))\
  .show()


+---------------+-----+
|          state|count|
+---------------+-----+
|        Gujarat|   35|
|         Punjab|   32|
|    Maharashtra|   30|
|             -1|   24|
|    West Bengal|   24|
|          Assam|   21|
|     Tamil Nadu|   20|
| Andhra Pradesh|   10|
|  Uttar Pradesh|    9|
|         Kerala|    8|
|         Odisha|    7|
|      Karnataka|    6|
|      Rajasthan|    6|
|      Telangana|    5|
|            Goa|    3|
|          Bihar|    3|
| Madhya Pradesh|    2|
|        Manipur|    2|
|Jammu & Kashmir|    2|
|       Nagaland|    1|
+---------------+-----+
only showing top 20 rows



In [19]:
#3.. How many dishes from state Karnataka? 

df.filter(col("state")=="Karnataka").count()

6

In [36]:
from pyspark.sql.functions import when, col
 
df = df.withColumn(
    "region",
    when(col("state").isin(
        "Andhra Pradesh", "Karnataka", "Kerala",
        "Tamil Nadu", "Telangana"
    ), "South")
    .when(col("state").isin(
        "Assam", "Manipur", "Meghalaya", "Mizoram",
        "Nagaland", "Sikkim", "Tripura"
    ), "North East")
    .when(col("state").isin(
        "Bihar", "Jharkhand", "Odisha", "West Bengal"
    ), "East")
    .when(col("state").isin(
        "Chhattisgarh", "Madhya Pradesh"
    ), "Central")
    .when(col("state").isin(
        "Goa", "Gujarat", "Maharashtra", "Rajasthan"
    ), "West")
    .when(col("state").isin(
        "Delhi", "Haryana", "Himachal Pradesh",
        "Jammu and Kashmir", "Punjab", "Uttar Pradesh",
        "Uttarakhand"
    ), "North")
    .otherwise("Unknown")
)
 
df.show()

+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+-------+
|          name|         ingredients|      diet|prep_time|cook_time|flavor_profile| course|        state| region|
+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+-------+
|    Balu shahi|Maida flour, yogu...|vegetarian|       45|       25|         sweet|dessert|  West Bengal|   East|
|        Boondi|Gram flour, ghee,...|vegetarian|       80|       30|         sweet|dessert|    Rajasthan|   West|
|Gajar ka halwa|Carrots, milk, su...|vegetarian|       15|       60|         sweet|dessert|       Punjab|  North|
|        Ghevar|Flour, ghee, kewr...|vegetarian|       15|       30|         sweet|dessert|    Rajasthan|   West|
|   Gulab jamun|Milk powder, plai...|vegetarian|       15|       40|         sweet|dessert|  West Bengal|   East|
|        Imarti|Sugar syrup, lent...|vegetarian|       10|       50|         sweet|desse

In [37]:
#4.List number of unique regions 
df.select("region").distinct().show()


+----------+
|    region|
+----------+
|   Unknown|
|     South|
|   Central|
|      East|
|      West|
|North East|
|     North|
+----------+



In [38]:
#5. Count number of dishes from each region. 

df.groupBy("region")\
 .count()\
 .orderBy(desc("count"))\
 .show()

+----------+-----+
|    region|count|
+----------+-----+
|      West|   74|
|     South|   49|
|     North|   43|
|      East|   34|
|   Unknown|   27|
|North East|   25|
|   Central|    3|
+----------+-----+



In [29]:
#6.List unique 'flavor_profile' and 'course' 

df.select("flavor_profile","course").distinct().show()

+--------------+-----------+
|flavor_profile|     course|
+--------------+-----------+
|        bitter|      snack|
|         spicy|    starter|
|          sour|main course|
|            -1|      snack|
|            -1|main course|
|         sweet|main course|
|        bitter|main course|
|         spicy|      snack|
|         sweet|    dessert|
|         spicy|main course|
+--------------+-----------+



In [30]:
#7.Which state has more 'main course'? 

df.filter(col("course") == "main course")\
 .groupBy("state")\
 .count()\
 .orderBy(desc("count"))\
 .show()

+---------------+-----+
|          state|count|
+---------------+-----+
|         Punjab|   28|
|     Tamil Nadu|   17|
|          Assam|   15|
|        Gujarat|   12|
|    Maharashtra|   12|
|             -1|    9|
|    West Bengal|    9|
|         Kerala|    5|
|      Karnataka|    4|
|      Rajasthan|    3|
|  Uttar Pradesh|    3|
|          Bihar|    2|
|       Nagaland|    1|
|         Odisha|    1|
| Madhya Pradesh|    1|
|        Manipur|    1|
|Jammu & Kashmir|    1|
|            Goa|    1|
|        Haryana|    1|
|   NCT of Delhi|    1|
+---------------+-----+
only showing top 20 rows



In [39]:
#8.Give the %of dishes from each region. 

total = df.count()

df.groupBy("region")\
 .count()\
 .withColumn("percentage",round((col("count")/total)*100 ,2))\
 .orderBy(desc("percentage"))\
 .show()

+----------+-----+----------+
|    region|count|percentage|
+----------+-----+----------+
|      West|   74|     29.02|
|     South|   49|     19.22|
|     North|   43|     16.86|
|      East|   34|     13.33|
|   Unknown|   27|     10.59|
|North East|   25|       9.8|
|   Central|    3|      1.18|
+----------+-----+----------+



In [ ]:
#9.caseList the states which has more dishes from each region.  
from pyspark.sql.window import Window
 
state_region_count = df.groupBy("region", "state").count()
 
window = Window.partitionBy("region").orderBy(desc("count"))
 
result = state_region_count \
    .withColumn("rank", rank().over(window)) \
    .filter(col("rank") == 1) \
    .drop("rank")
 
result.show()

+----------+--------------+-----+
|    region|         state|count|
+----------+--------------+-----+
|   Central|Madhya Pradesh|    2|
|      East|   West Bengal|   24|
|     North|        Punjab|   32|
|North East|         Assam|   21|
|     South|    Tamil Nadu|   20|
|   Unknown|            -1|   24|
|      West|       Gujarat|   35|
+----------+--------------+-----+

